# BabyMoE training on Colab

This notebook only orchestrates versioned scripts. Large artifacts persist in Google Drive, and every command is safe to rerun after a disconnected session.

In [ ]:
GITHUB_REPO = "https://github.com/Amos-Luna/BabyMoE_Research.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/BabyMoE"
LOCAL_ARTIFACTS = "/content/babymoe_artifacts"
EXPERIMENTS = [
    ("configs/moe_learned.yaml", [13, 42, 71]),
    ("configs/dense_match.yaml", [13, 42, 71]),
    ("configs/moe_frozen_gate.yaml", [42]),
]
STOP_AT_EXPOSURE = 10_000_000
CONTINUE_SEED42_TO_EXPOSURE = 100_000_000
AUTO_CONTINUE_IF_PILOT_PASSES = True
RUN_SMOKE = True

In [ ]:
import json, os, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")
repo_dir = Path("/content/BabyMoE_Research")
runs_root = Path(DRIVE_ROOT) / "artifacts" / "runs"
recorded_shas = {
    json.loads(path.read_text())["git_sha"]
    for path in runs_root.glob("*_seed*/run_meta.json")
}
if len(recorded_shas) > 1:
    raise RuntimeError(f"Persistent runs contain multiple Git SHAs: {recorded_shas}")
resume_ref = next(iter(recorded_shas), None)
if not (repo_dir / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, GITHUB_REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin"], check=True)
if resume_ref:
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", resume_ref], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", BRANCH], check=True)
os.chdir(repo_dir)
for secret in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        value = userdata.get(secret)
        if value:
            os.environ[secret] = value
    except Exception:
        pass
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("Add HF_TOKEN to Colab Secrets after accepting the gated BabyLM datasets.")
print(f"Repository ready at {repo_dir}; exact-resume ref={resume_ref or BRANCH}")

In [ ]:
bootstrap = ["bash", "scripts/colab_bootstrap.sh", "--drive-root", DRIVE_ROOT]
if RUN_SMOKE:
    bootstrap.append("--run-smoke")
subprocess.run(bootstrap, check=True, env=os.environ.copy())

In [ ]:
subprocess.run([
    "python", "scripts/prepare_data.py",
    "--config", EXPERIMENTS[0][0],
    "--artifacts-root", f"{DRIVE_ROOT}/artifacts",
], check=True, env=os.environ.copy())

In [ ]:
# Copy packed blocks once to Colab SSD; tokenizer, runs, and checkpoints remain on Drive.
subprocess.run([
    "bash", "scripts/stage_for_training.sh",
    f"{DRIVE_ROOT}/artifacts", LOCAL_ARTIFACTS,
], check=True)

In [ ]:
# Representative A100 throughput and wall-time estimate before long runs.
for config, _ in EXPERIMENTS[:2]:
    subprocess.run([
        "python", "scripts/train.py", "--config", config,
        "--artifacts-root", LOCAL_ARTIFACTS,
        "--dry-run", "--benchmark-steps", "3",
    ], check=True, env=os.environ.copy())

In [ ]:
for config, seeds in EXPERIMENTS:
    for seed in seeds:
        print(f"\n=== {config} | seed={seed} | target={STOP_AT_EXPOSURE:,} ===", flush=True)
        subprocess.run([
            "python", "scripts/train.py",
            "--config", config,
            "--seed", str(seed),
            "--stop-at-exposure", str(STOP_AT_EXPOSURE),
            "--artifacts-root", LOCAL_ARTIFACTS,
        ], check=True, env=os.environ.copy())

In [ ]:
# Controlled 10M probes and preregistered three-seed decision gate.
runs_root = Path(DRIVE_ROOT) / "artifacts" / "runs"
analysis_paths = []
for seed in (13, 42, 71):
    moe_run = runs_root / f"moe_learned_seed{seed}"
    dense_run = runs_root / f"dense_match_seed{seed}"
    subprocess.run([
        "python", "scripts/analyze_routing.py",
        "--run-dir", str(moe_run),
        "--dense-run-dir", str(dense_run),
        "--milestone", "10",
    ], check=True, env=os.environ.copy())
    analysis_paths.append(str(moe_run / "analysis" / "analysis_10M.json"))
subprocess.run([
    "python", "scripts/summarize_pilots.py",
    *sum((["--analysis", path] for path in analysis_paths), []),
    "--output", str(runs_root / "pilot_summary.json"),
], check=True)
subprocess.run([
    "python", "scripts/analyze_routing.py",
    "--run-dir", str(runs_root / "moe_frozen_gate_seed42"),
    "--dense-run-dir", str(runs_root / "dense_match_seed42"),
    "--milestone", "10",
], check=True, env=os.environ.copy())

In [ ]:
# Exact seed-42 continuation reuses the saved optimizer, stream, and RNG states.
pilot_gate = json.loads((runs_root / "pilot_summary.json").read_text())
if AUTO_CONTINUE_IF_PILOT_PASSES and pilot_gate["proceed_to_100M"]:
    for config, _ in EXPERIMENTS[:2]:
        subprocess.run([
            "python", "scripts/train.py",
            "--config", config,
            "--seed", "42",
            "--stop-at-exposure", str(CONTINUE_SEED42_TO_EXPOSURE),
            "--artifacts-root", LOCAL_ARTIFACTS,
            "--resume",
        ], check=True, env=os.environ.copy())
else:
    print("100M continuation paused by the pilot gate:", pilot_gate)

In [ ]:
import json
runs_root = Path(DRIVE_ROOT) / "artifacts" / "runs"
for summary_path in sorted(runs_root.glob("*_seed*/summary.json")):
    summary = json.loads(summary_path.read_text())
    print(summary_path.parent.name, "exposure=", f"{summary['actual_exposure_units']:,.0f}", "steps=", summary["global_step"], "milestones=", summary["saved_milestones"][-3:])